# Script de Monitoreo de Acciones con WhatsApp

Este notebook contiene un script para enviar información de precios de acciones por WhatsApp cada hora.

## Características:
- Monitoreo de acciones chilenas (Bolsa de Santiago)
- Monitoreo de acciones estadounidenses (Yahoo Finance)
- Envío automático por WhatsApp cada hora
- Fácil configuración de símbolos a monitorear

In [ ]:
# Importar todas las librerías necesarias
import yfinance as yf
import pywhatkit as pwk
import schedule
import time
import requests
import pandas as pd
from datetime import datetime
import json

# Importación para web scraping
try:
    from bs4 import BeautifulSoup
    print("✅ BeautifulSoup disponible - Investing.com habilitado para Chile")
except ImportError:
    print("⚠️  Para usar Investing.com instala: pip install beautifulsoup4")
    print("   Mientras tanto se usará Yahoo Finance para Chile")

In [ ]:
# CONFIGURACIÓN - Personaliza estos valores
NUMERO_WHATSAPP = "+56995959596"  

# Acciones estadounidenses (símbolos de Yahoo Finance)
ACCIONES_USA = [
    "AAPL",   
    "MSFT",   
    "GOOGL",  
    "AMZN",   
    "TSLA",   
    "NVDA",   
    "NBIS",   
]

# Acciones chilenas - TOP 15 IPSA (símbolos de Yahoo Finance terminan en .SN)
ACCIONES_CHILE = [
    "SQM-B.SN",     
    "COPEC.SN",     
    "CHILE.SN",     
    "FALABELLA.SN", 
    "CENCOSUD.SN",  
    "LTM.SN",       
    "BCI.SN",       
    "COLBUN.SN",    
    "ENELAM.SN",    
    "CCU.SN",       
    "PARAUCO.SN",   
    "AGUAS-A.SN",   
    "MALLPLAZA.SN",   
    "ABC.SN",     
    "BSANTANDER.SN",
    "CAP.SN",
    "CMPC.SN",
    "CONCHATORO.SN",
    "ILC.SN",
    "QUINENCO.SN",
    "CFIETFIPSA.SN",
    "CFINASDAQ.SN",
    "FALABELLA.SN",
    "HABITAT.SN",
    "ITAUCL.SN",
    "LIPIGAS.SN",
    "VAPORES.SN",
]

In [ ]:
def obtener_precios_usa(simbolos):
    """Obtiene precios de acciones estadounidenses usando Yahoo Finance"""
    precios = {}
    
    for simbolo in simbolos:
        try:
            ticker = yf.Ticker(simbolo)
            info = ticker.info
            precio_actual = info.get('currentPrice', 'N/A')
            cambio_pct = info.get('regularMarketChangePercent', 0)
            
            precios[simbolo] = {
                'precio': precio_actual,
                'cambio_pct': cambio_pct,
                'moneda': 'USD'
            }
        except Exception as e:
            print(f"Error obteniendo precio de {simbolo}: {e}")
            precios[simbolo] = {'precio': 'Error', 'cambio_pct': 0, 'moneda': 'USD'}
    
    return precios

In [ ]:
def obtener_precios_chile(simbolos):
    """Obtiene precios de acciones chilenas usando Yahoo Finance (versión estable)"""
    precios = {}
    
    for simbolo in simbolos:
        try:
            ticker = yf.Ticker(simbolo)
            
            # Intentar obtener datos intraday primero (más actualizados)
            try:
                hist = ticker.history(period="1d", interval="5m")
                if not hist.empty:
                    precio_actual = hist['Close'].iloc[-1]
                    precio_apertura = hist['Open'].iloc[0]
                    cambio_pct = ((precio_actual - precio_apertura) / precio_apertura * 100) if precio_apertura != 0 else 0
                    fuente_detalle = "Yahoo Finance (5min)"
                else:
                    raise Exception("Sin datos intraday")
                    
            except:
                # Fallback a datos diarios
                hist = ticker.history(period="2d")
                if not hist.empty:
                    precio_actual = hist['Close'].iloc[-1]
                    precio_anterior = hist['Close'].iloc[-2] if len(hist) > 1 else precio_actual
                    cambio_pct = ((precio_actual - precio_anterior) / precio_anterior * 100) if precio_anterior != 0 else 0
                    fuente_detalle = "Yahoo Finance (daily)"
                else:
                    raise Exception("Sin datos disponibles")
            
            # Validar que el precio sea razonable
            if precio_actual > 0:
                precios[simbolo] = {
                    'precio': round(float(precio_actual), 2),
                    'cambio_pct': round(float(cambio_pct), 2),
                    'moneda': 'CLP',
                    'fuente': fuente_detalle
                }
                
                # Log mejorado
                simbolo_corto = simbolo.replace('.SN', '')
                precio_formateado = f"${precio_actual:,.0f}" if precio_actual >= 1000 else f"${precio_actual:.2f}"
                cambio_emoji = "📈" if cambio_pct > 0 else "📉" if cambio_pct < 0 else "➡️"
                print(f"✅ {simbolo_corto}: {precio_formateado} CLP ({cambio_pct:+.2f}%) {cambio_emoji}")
            else:
                raise Exception("Precio no válido")
                
        except Exception as e:
            print(f"❌ Error obteniendo precio de {simbolo.replace('.SN', '')}: {str(e)}")
            precios[simbolo] = {
                'precio': 'Error', 
                'cambio_pct': 0, 
                'moneda': 'CLP', 
                'fuente': 'Error'
            }
    
    return precios

In [ ]:
def crear_mensaje_precios(precios_usa, precios_chile):
    """Crea el mensaje de WhatsApp con los precios"""
    mensaje = f"*REPORTE DE ACCIONES* - {datetime.now().strftime('%d/%m/%Y %H:%M')}\n\n"
    
    # Acciones USA
    mensaje += "🇺🇸 *ACCIONES USA*\n"
    mensaje += "─" * 25 + "\n"
    
    for simbolo, data in precios_usa.items():
        precio = data['precio']
        cambio = data['cambio_pct']
        emoji = "📈" if cambio > 0 else "📉" if cambio < 0 else "➡️"
        
        # Formatear precio, manejando casos de error
        if isinstance(precio, (int, float)) and precio != 'N/A' and precio != 'Error':
            precio_formateado = f"${precio:.2f}"
        else:
            precio_formateado = f"${precio}"
        
        mensaje += f"{emoji} *{simbolo}*: {precio_formateado} USD ({cambio:+.2f}%)\n"
    
    # Acciones Chile
    mensaje += "\n🇨🇱 *ACCIONES CHILE*\n"
    mensaje += "─" * 25 + "\n"
    
    for simbolo, data in precios_chile.items():
        precio = data['precio']
        cambio = data['cambio_pct']
        emoji = "📈" if cambio > 0 else "📉" if cambio < 0 else "➡️"
        simbolo_limpio = simbolo.replace('.SN', '')
        
        # Formatear precio con comas para miles, decimales para precios menores a 1000
        if isinstance(precio, (int, float)) and precio != 'N/A' and precio != 'Error':
            if precio >= 1000:
                precio_formateado = f"${precio:,.0f}"  # Sin decimales para miles: $1,234
            else:
                precio_formateado = f"${precio:.2f}"   # Con decimales para < 1000: $21.53
        else:
            precio_formateado = f"${precio}"
        
        mensaje += f"{emoji} *{simbolo_limpio}*: {precio_formateado} CLP ({cambio:+.2f}%)\n"
    
    mensaje += f"\n _Actualizado automáticamente_"
    
    return mensaje

In [ ]:
def enviar_whatsapp_instantaneo(numero, mensaje):
    """Envía mensaje de WhatsApp de forma instantánea usando pywhatkit"""
    try:
        # Obtener hora actual + 1 minuto para envío inmediato
        ahora = datetime.now()
        hora = ahora.hour
        minuto = ahora.minute + 1
        
        # Ajustar si los minutos superan 59
        if minuto >= 60:
            minuto = minuto - 60
            hora = hora + 1
            if hora >= 24:
                hora = 0
        
        print(f"Programando envío para {hora:02d}:{minuto:02d}")
        
        # Enviar mensaje
        pwk.sendwhatmsg(numero, mensaje, hora, minuto, 15, True, 3)
        print("✅ Mensaje enviado exitosamente")
        
    except Exception as e:
        print(f"❌ Error enviando WhatsApp: {e}")
        return False
    
    return True

In [ ]:
def ejecutar_reporte():
    """Función principal que obtiene precios y envía el reporte"""
    print(f"🔄 Iniciando reporte de acciones - {datetime.now().strftime('%H:%M:%S')}")
    
    # Obtener precios
    print("📊 Obteniendo precios de acciones USA...")
    precios_usa = obtener_precios_usa(ACCIONES_USA)
    
    print("📊 Obteniendo precios de acciones Chile...")
    precios_chile = obtener_precios_chile(ACCIONES_CHILE)
    
    # Crear mensaje
    mensaje = crear_mensaje_precios(precios_usa, precios_chile)
    
    # Mostrar mensaje en consola
    print("\n" + "="*50)
    print("MENSAJE A ENVIAR:")
    print("="*50)
    print(mensaje)
    print("="*50 + "\n")
    
    # Enviar WhatsApp
    print("📱 Enviando reporte por WhatsApp...")
    exito = enviar_whatsapp_instantaneo(NUMERO_WHATSAPP, mensaje)
    
    if exito:
        print("✅ Reporte completado exitosamente")
    else:
        print("❌ Error en el envío del reporte")
    
    print("-" * 50)

In [ ]:
# SISTEMA AUTOMÁTICO

def iniciar_monitoreo_automatico():
    """Inicia el monitoreo automático con envíos cada 15 minutos"""
    ejecutar_reporte()  # Ejecutar inmediatamente al iniciar
    
    # Programar reportes cada 15 minutos durante horario de mercado (9:00 AM - 4:00 PM)
    horas_mercado = []
    
    # Generar horarios cada 15 minutos desde 09:00 hasta 16:00
    for hora in range(9, 17):  # De 9 AM a 4 PM (16:00)
        for minuto in [0, 30]:
            tiempo = f"{hora:02d}:{minuto:02d}"
            horas_mercado.append(tiempo)
    
    for hora in horas_mercado:
        schedule.every().day.at(hora).do(ejecutar_reporte)
    
    print("⏰ Sistema automático configurado!")
    print("📅 Reportes cada 15 minutos desde 09:00 hasta 16:00")
    print(f"🕒 Total de reportes por día: {len(horas_mercado)}")
    print("📋 Horarios programados:")

    for i, hora in enumerate(horas_mercado, 1):
        if i % 8 == 1:  # Nueva línea cada 8 horarios
            print(f"\n   {hora}", end="")
        else:
            print(f"  {hora}", end="")
    
    print("\n\n🔄 Iniciando monitoreo continuo...")
    print("⚠️  Para detener, interrumpe la ejecución de la celda (Ctrl+C)")
    print("-" * 60)
    

    while True:
        schedule.run_pending()
        time.sleep(60)  # Verificar cada minuto

iniciar_monitoreo_automatico()